In [1]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

import equinox as eqx
import equinox.nn as nn
import snnax.snn as snn

import optax

In [ ]:
# from eleanor.datasets import shuffle, CoordinatesToTFS, CoordinatesToRate, YinYangDataset

In [ ]:
# Defining functions in here for now, will fix

def shuffle(dataset, shuffle_rng, batch_size):
    x, y = dataset

    cutoff = y.shape[0] % batch_size

    obs = jax.random.permutation(shuffle_rng, x, axis=0)[:-cutoff]
    labels = jax.random.permutation(shuffle_rng, y, axis=0)[:-cutoff]

    obs = jnp.reshape(obs, (-1, batch_size) + obs.shape[1:])
    labels = jnp.reshape(labels, (-1, batch_size))  # should make batch size a global

    return (obs, labels)

def CoordinatesToTFS(seq_length, t_early, t_late, dt = 1e-3):

    def forward(key, coordinate_values):
        times = t_early + coordinate_values * (t_late - t_early)
        batch_size, num_channels = times.shape

        indices = jnp.stack([
            jnp.repeat(jnp.arange(times.shape[0]), times.shape[1]), # Batch
            jnp.round(times.flatten() / dt),                        # Time
            jnp.tile(jnp.arange(times.shape[1]), times.shape[0])    # Channels
        ]).astype(int)
        spikes = jnp.zeros((batch_size, seq_length, num_channels)).at[indices[0,:], indices[1,:], indices[2,:]].set(1)
        return spikes.astype(jnp.uint8)

    return forward

def CoordinatesToRate(seq_length, rate):

    def forward(key, coordinate_values):
        unrolled_data = jnp.repeat(coordinate_values[:,jnp.newaxis,:], seq_length, axis=1)
        return jax.random.bernoulli(key, unrolled_data*rate).astype(jnp.uint8)

    return forward

def YinYangDataset(r_small = 0.1, r_big = 0.5, size = 1000, seed = 42):
    "Adapted from https://github.com/lkriener/yin_yang_data_set"
    key = jax.random.key(seed)

    goal_class = jax.random.randint(key, (size,), 0, 3)

    @jax.jit
    def dist_to_right_dot(x, y):
        return jnp.sqrt((x - 1.5 * r_big)**2 + (y - r_big)**2)

    @jax.jit
    def dist_to_left_dot(x, y):
        return jnp.sqrt((x - 0.5 * r_big)**2 + (y - r_big)**2)

    def which_class(x, y):
        # equations inspired by
        # https://link.springer.com/content/pdf/10.1007/11564126_19.pdf
        d_right = dist_to_right_dot(x, y)
        d_left = dist_to_left_dot(x, y)
        criterion1 = d_right <= r_small
        criterion2 = jnp.logical_and(d_left > r_small, d_left <= 0.5 * r_big)
        criterion3 = jnp.logical_and(y > r_big, d_right > 0.5 * r_big)
        is_yin = jnp.logical_or(jnp.logical_or(criterion1, criterion2), criterion3)
        is_circles = jnp.logical_or(d_right < r_small, d_left < r_small)
        return jax.lax.cond(is_circles, lambda x: 2, lambda x: x, is_yin.astype(int))

        if is_circles:
            return 2
        return int(is_yin)

    def getSample(key, goal):
        def condition(state):
            key, x, y, c = state
            cond = jnp.sqrt((x - r_big)**2 + (y - r_big)**2) > r_big
            return jnp.logical_or(c != goal, cond)

        def body(state):
            key, x, y, c = state
            key, subkey = jax.random.split(key)
            x, y = jax.random.uniform(subkey, (2,)) * 2. * r_big
            c = which_class(x, y)
            return (key, x, y, c)

        init_val = -1
        key, x, y, c = jax.lax.while_loop(condition, body, (key, init_val, init_val, init_val))

        return key, (x, y, c)

    _, (x,y,c) = jax.lax.scan(getSample, key, goal_class)

    return jnp.stack([x,y,1-x,1-y], axis=1), c

# # fig, axes = plt.subplots(ncols=3, sharey=True, figsize=(15, 8))
# # titles = ['Training set', 'Validation set', 'Test set']
# # for i, size in enumerate([5000, 1000, 1000]):
# #     data, c = YinYangDataset(size=size)
# #     x = data[:,0]
# #     y = data[:,1]

# #     idx0 = jnp.where(c==0)
# #     idx1 = jnp.where(c==1)
# #     idx2 = jnp.where(c==2)

# #     axes[i].set_title(titles[i])
# #     axes[i].set_aspect('equal', adjustable='box')
# #     axes[i].scatter(x[idx0],y[idx0], color='C0', edgecolor='k', alpha=0.3)
# #     axes[i].scatter(x[idx1],y[idx1], color='C1', edgecolor='k', alpha=0.3)
# #     axes[i].scatter(x[idx2],y[idx2], color='C2', edgecolor='k', alpha=0.3)
# #     axes[i].set_xlabel('x1')
# #     if i == 0:
# #         axes[i].set_ylabel('y1')
# # plt.show()

# # transform = CoordinatesToRate(100, 0.1)
# # spikesRate = transform(jax.random.key(0), data)

# # transform = CoordinatesToTFS(100, 0.0, 100.0, dt=1e-3)
# # spikesTFS = transform(jax.random.key(0), data)

# # _, ax = plt.subplots(1,2, figsize=(12,4))
# # ax[0].plot(*jnp.where(jnp.concatenate(spikesRate[200:210])), '|', markersize=20)
# # [ax[0].axvline(x*100, c='r') for x in range(10)]
# # ax[0].grid()
# # ax[0].set_title('Rate encoding')

# # ax[1].plot(*jnp.where(jnp.concatenate(spikesTFS[200:210])), '|', markersize=20)
# # [ax[1].axvline(x*100, c='r') for x in range(10)]
# # ax[1].grid()
# # ax[1].set_title('First-spike encoding')

# # plt.show()


# # train_size, test_size = 5000, 1000

# # # Generate the dataset
# # x_train, y_train = YinYangDataset(size=train_size)
# # x_test, y_test = YinYangDataset(size=test_size)

# # print("Generated x_train shape:", x_train.shape)
# # print("Generated y_train shape:", y_train.shape)


In [ ]:
# Generating the data

training_data, training_classes = YinYangDataset(size=5000)
validation_data, validation_classes = YinYangDataset(size=5000)
test_data, test_classes = YinYangDataset(size=1000)

YYtrainset = (training_data, training_classes)
YYtestset = (test_data, test_classes)

In [ ]:
from eleanor.models import Heracles, Scaler, FeLIF
from chex import Array, PRNGKey
from typing import Optional

key = jax.random.key(0)
key1, key2, key3, key4, key5, key6 = jax.random.split(key, 6)


class EncodingLayer(eqx.Module):

    gain: Array
    bias: Array
    expansion: float

    def __init__(self, gain: Array, bias: Array, expansion: float) -> None:
        self.gain = gain
        self.bias = bias
        self.expansion = expansion

    def __call__(self, synaptic_input: Array, *, key: Optional[PRNGKey] = None):
        output = self.gain * (jnp.tile(synaptic_input, self.expansion) + self.bias)
        return output


enc_gain = jax.random.normal(key1, shape=(128,)) * 0.18436009935019085
enc_bias = jax.random.normal(key2, shape=(128,))
model = snn.Sequential(
    EncodingLayer(enc_gain, enc_bias, 32),
    nn.Linear(128, 256, key=key3),
    snn.LIF([0.9, 0.8], key=key4),
    nn.Linear(256, 3, key=key5),
    # snn.LIF([.9, .8], key=key6)
    Scaler(1000, 1),
    FeLIF(dt=1e-3, V_thr=0.5, paramsScale=1e12),
)
model

In [ ]:
from functools import partial
from jax.tree_util import tree_map


# Simple batched loss function
@partial(jax.vmap, in_axes=(None, None, 0, 0, 0))
def loss_fn(model, in_states, in_spikes, tgt_class, key):
    out_state, out_spikes = model(in_states, in_spikes, key=key)

    # Get the output of last layer
    final_layer_out = out_spikes[-1][0]
    # final_layer_out = out_spikes[-1]

    # Sum all spikes in each output neuron along time axis
    # pred = tree_map(lambda x: jnp.sum(x, axis=0), final_layer_out)
    pred = final_layer_out.sum(axis=0)

    target = jax.nn.one_hot(tgt_class, 3)
    loss = optax.softmax_cross_entropy(pred, target)
    return loss


# Calculating the gradient with Equinox PyTree filters and
# subsequently jitting the resulting function
@eqx.filter_value_and_grad
def loss_and_grad(model, in_states, in_spikes, tgt_class, key):
    keys = jax.random.split(key, 128)
    return jnp.mean(loss_fn(model, in_states, in_spikes, tgt_class, keys))


@partial(jax.vmap, in_axes=(None, None, 0, 0, 0))
def accuracy_fn(model, in_states, in_spikes, tgt_class, key):
    out_state, out_spikes = model(in_states, in_spikes, key=key)
    final_layer_out = out_spikes[-1][0]
    # final_layer_out = out_spikes[-1]
    pred = final_layer_out.sum(axis=0)
    predicted_class = jnp.argmax(pred)
    return predicted_class == tgt_class


@eqx.filter_jit
def calc_accuracy(model, in_states, in_spikes, tgt_class, key):
    keys = jax.random.split(key, 128)
    accuracy = accuracy_fn(model, in_states, in_spikes, tgt_class, keys)
    return jnp.mean(accuracy)


# Finally, we update the parameters using a simple optimizer
@eqx.filter_jit
def update(model, in_states, opt_state, in_spikes, tgt_class, key):
    # Get gradients
    loss, grads = loss_and_grad(model, in_states, in_spikes, tgt_class, key)

    # Calculate parameter updates using the optimizer
    updates, opt_state = optim.update(grads, opt_state)

    # Update parameter PyTree with Equinox and optax
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

In [ ]:
nb_outputs = 3 # 3 classes in the YY dataset
nb_channels = YYtrainset[0].shape[1] # Number of different input features
nb_steps = 100
time_step = 1e-3

YYtrainset = YinYangDataset(size=5000)
YYtestset = YinYangDataset(size=1000)

# Rate encoding
# transform = CoordinatesToRate(seq_length=100, rate=0.1)  # Adjust `rate` as needed
# encoded_train_data = transform(jax.random.key(0), YYtrainset[0])
# encoded_test_data = transform(jax.random.key(0), YYtestset[0])

# TFS encoding
transform = CoordinatesToTFS(100, 0.0, 100.0, dt=1)
encoded_train_data = transform(jax.random.key(0), YYtrainset[0])
encoded_test_data = transform(jax.random.key(0), YYtestset[0])

# Set the train and test sets with encoded data
trainset = (encoded_train_data, YYtrainset[1])
testset = (encoded_test_data, YYtestset[1])


In [ ]:
from tqdm import trange

optim = optax.adamax(learning_rate=1e-3, b1=0.9, b2=0.995)
opt_state = optim.init(eqx.filter(model, eqx.is_inexact_array))

initial_state = model.init_state(in_shape=(4,), key=jax.random.key(0))
total_loss = []
total_accuracy = []

pbar = trange(0, 30)
for epoch in pbar:
    key, epoch_key = jax.random.split(key)
    x_train, y_train = shuffle(trainset, epoch_key, 128)
    loss_train = []
    for in_spikes, tgt_class in zip(x_train, y_train):
        # Initializing the membrane potentials of LIF neurons
        model, opt_state, loss = update(
            model, initial_state, opt_state, in_spikes, tgt_class, key
        )
        loss_train.append(loss)
    loss_train = jnp.mean(jnp.asarray(loss_train))
    total_loss.append(loss_train)

    x_test, y_test = shuffle(testset, jax.random.key(0), 128)
    accuracy_test = []
    for in_spikes, tgt_class in zip(x_test, y_test):
        # Initializing the membrane potentials of LIF neurons
        accuracy = calc_accuracy(
            model, initial_state, in_spikes, tgt_class, jax.random.key(0)
        )
        accuracy_test.append(accuracy)
    accuracy_test = jnp.mean(jnp.asarray(accuracy_test))
    total_accuracy.append(accuracy_test)


fig, axs = plt.subplots()
axs.plot(total_loss, color='red')
axs.twinx().plot(total_accuracy)

plt.show()

In [ ]:
# jnp.save("results/braillenLossFeLIF_Bruno.npy", total_loss)
# jnp.save("results/brailleAccFeLIF_Bruno.npy", total_accuracy)